# Calling a local model from Python

Before you run anything:

1. LM Studio is open, a chat (instruct) model is loaded, and the server is running on port 1234 (see part 4 of this module's README).
2. You ran `uv add openai matplotlib` in module 00.
3. This notebook's kernel is the `.venv` inside `ML_Lab` (top right of VS Code).

You'll use the official `openai` Python package. It doesn't care whose server it talks to, only that the server speaks the same API. That's why the same code works against OpenAI, LM Studio, Ollama, vLLM, and most hosted open-model providers.

## 1. Connect

In [ ]:
from openai import OpenAI

# LM Studio ignores the API key, but the client requires one, so any string works.
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

models = client.models.list()
for m in models.data:
    print(m.id)

That's every model LM Studio can serve. If you see a connection error instead, the server isn't running.

In [ ]:
# LM Studio ships with an embedding model that shows up in this list but can't chat, so skip it.
chat_models = [m.id for m in models.data if "embed" not in m.id]
MODEL = chat_models[0]  # if this picks the wrong one, paste the exact id of your loaded model here
print("Using:", MODEL)

## 2. Your first request

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In two sentences, what is quantization in machine learning?"},
    ],
    temperature=0.7,
    max_tokens=200,
)

print(response.choices[0].message.content)
print()
print(response.usage)

What came back:

- `choices[0].message.content` is the text.
- `usage` counts tokens. `prompt_tokens` is what you sent and `completion_tokens` is what the model generated. Hosted APIs bill on exactly these two numbers, so get used to watching them.

`messages` is a list of turns, each with a role. `system` sets behavior, `user` is the human, and `assistant` is the model's earlier replies.

## 3. The model has no memory

This is the most misunderstood thing about LLM APIs. The model is stateless. Every request is independent, and the only "memory" is whatever history you send back in `messages`. Chat apps feel like they remember because the app resends the whole conversation every time.

In [ ]:
# Request 1: tell it something
first = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Pujan and I'm building an app called Darvis."}],
    max_tokens=100,
)
reply = first.choices[0].message.content
print("Model:", reply)

# Request 2: a brand new request with no history
forgetful = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What's my name and what am I building?"}],
    max_tokens=100,
)
print("\nWithout history:", forgetful.choices[0].message.content)

# Request 3: same question, but we send the history ourselves
history = [
    {"role": "user", "content": "My name is Pujan and I'm building an app called Darvis."},
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "What's my name and what am I building?"},
]
remembers = client.chat.completions.create(model=MODEL, messages=history, max_tokens=100)
print("\nWith history:", remembers.choices[0].message.content)

This is also why long conversations get slower and more expensive: the prompt grows with every turn.

## 4. Streaming and speed

Chat UIs print tokens as they arrive instead of waiting for the full answer. With `stream=True` you get chunks back one at a time. This cell also measures the two numbers that matter when you serve a model to users: time to first token (how long before anything appears) and tokens per second.

In [ ]:
import time

start = time.perf_counter()
first_token_time = None
chunks = 0

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what a neural network is to a high school student in about 100 words."}],
    stream=True,
    max_tokens=250,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        chunks += 1
        print(delta, end="", flush=True)

end = time.perf_counter()
print(f"\n\nTime to first token: {first_token_time - start:.2f}s")
print(f"Approx tokens/sec:   {chunks / (end - first_token_time):.1f}  (one chunk is roughly one token)")

## 5. Temperature, measured

Same prompt, five samples at each temperature. Count how many distinct answers you get.

In [ ]:
prompt = "Give me a name for a coffee shop run by robots. Reply with the name only."

for temp in [0.0, 0.7, 1.5]:
    outputs = []
    for _ in range(5):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=20,
        )
        outputs.append(r.choices[0].message.content.strip())
    print(f"temperature={temp}: {len(set(outputs))} distinct of 5")
    for o in outputs:
        print("   ", o)

Under the hood the model produces a probability for every token in its vocabulary. Temperature divides the raw scores (logits) before they're turned into probabilities. Low temperature sharpens the distribution so the top token almost always wins. High temperature flattens it so unlikely tokens get picked more often. You'll implement exactly this in module 06.

Rules of thumb: use 0 to 0.3 for extraction, classification, and code, and 0.7 to 1.0 for writing and brainstorming.

## 6. A helper you'll reuse

In [ ]:
def chat(prompt, system=None, temperature=0.7, max_tokens=400, model=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content

print(chat("What is 17 * 23? Show your work briefly.", temperature=0))

---
# Exercises

Try each one before scrolling to the solutions.

**Exercise 1: Benchmark.** Write `benchmark(model_id, runs=3)` that sends the same prompt `runs` times without streaming and returns the average generation speed in tokens per second, using `response.usage.completion_tokens` and a timer. Load a 4-bit and an 8-bit version of the same model in LM Studio and compare them.

**Exercise 2: A mini Darvis.** Put this data in a system prompt and ask the model questions about it:

```
CS 3114 | Prof. A | avg GPA 2.9 | A rate 31%
CS 3114 | Prof. B | avg GPA 3.3 | A rate 48%
CS 3214 | Prof. C | avg GPA 2.7 | A rate 22%
```

Ask "Which CS 3114 professor has better outcomes?" Then ask about CS 4104, which isn't in the data. Does it make something up? Change the system prompt so it says it doesn't know instead. This is grounding, the core idea behind retrieval-augmented generation (RAG). Real RAG adds a search step that picks which rows go into the prompt.

**Exercise 3: Structured output.** Your backend usually needs JSON, not prose. Ask the model to extract `{"course": str, "professor": str, "question_type": str}` from the sentence "is professor b easier than prof a for cs 3114". First try asking for JSON in the prompt and parsing it with `json.loads`. Then try LM Studio's `response_format` with a JSON schema, which forces the output to match.

**Exercise 4: Same code, different server.** Point a second client at Ollama (`http://localhost:11434/v1`) and run `chat()` against a model from `ollama list`. Nothing else in your code should change.

---
# Solutions

In [ ]:
# Exercise 1
def benchmark(model_id, runs=3, prompt="Write a 150-word paragraph about the history of basketball."):
    speeds = []
    for _ in range(runs):
        start = time.perf_counter()
        r = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=300,
        )
        elapsed = time.perf_counter() - start
        speeds.append(r.usage.completion_tokens / elapsed)
    return sum(speeds) / len(speeds)

# This number includes time to first token, so it reads a bit lower than LM Studio's own stat.
print(f"{MODEL}: {benchmark(MODEL):.1f} tokens/sec")

In [ ]:
# Exercise 2
DATA = '''
CS 3114 | Prof. A | avg GPA 2.9 | A rate 31%
CS 3114 | Prof. B | avg GPA 3.3 | A rate 48%
CS 3214 | Prof. C | avg GPA 2.7 | A rate 22%
'''

system = (
    "You answer questions about Virginia Tech courses using ONLY the data below. "
    "If the data doesn't contain the answer, say you don't have data for that course. "
    "Never guess.\n\nDATA:\n" + DATA
)

print(chat("Which CS 3114 professor has better outcomes?", system=system, temperature=0))
print("---")
print(chat("Which CS 4104 professor should I take?", system=system, temperature=0))

In [ ]:
# Exercise 3, approach 1: ask nicely, then parse
import json

question = "is professor b easier than prof a for cs 3114"

raw = chat(
    f'Extract fields from this question. Reply with JSON only, no other text, using keys '
    f'"course", "professor", "question_type".\n\nQuestion: {question}',
    temperature=0,
)
print("Raw:", raw)

try:
    print("Parsed:", json.loads(raw))
except json.JSONDecodeError:
    # Small models often wrap JSON in ```json fences. That's why approach 2 exists.
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    print("Parsed after cleanup:", json.loads(cleaned))

In [ ]:
# Exercise 3, approach 2: force the shape with a JSON schema
schema = {
    "type": "object",
    "properties": {
        "course": {"type": "string"},
        "professor": {"type": "string"},
        "question_type": {"type": "string", "enum": ["comparison", "recommendation", "requirements", "other"]},
    },
    "required": ["course", "professor", "question_type"],
}

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the fields from this question: {question}"}],
    response_format={"type": "json_schema", "json_schema": {"name": "query", "schema": schema}},
    temperature=0,
)
print(json.loads(r.choices[0].message.content))

In [ ]:
# Exercise 4
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
print([m.id for m in ollama.models.list().data])

OLLAMA_MODEL = ollama.models.list().data[0].id
r = ollama.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)
print(r.choices[0].message.content)